## Metadata dataset: used for Retrieval Agent (FAISS), Ranking Agent, Pricing Agent, Final Recommendation Card

In [1]:
# Install required packages
!pip install --quiet google-cloud-bigquery google-cloud-storage pandas datasets

# Imports
import pandas as pd
from datasets import load_dataset
from google.cloud import bigquery
from google.oauth2 import service_account
import os

# Initialize BigQuery client
project_id = "linear-theater-436300-r9"

client = bigquery.Client(project=project_id)

print("BigQuery client initialized for project:", project_id)


BigQuery client initialized for project: linear-theater-436300-r9


In [8]:
import json
from huggingface_hub import hf_hub_download

# Download item metadata (1M sample)
filepath = hf_hub_download(
    repo_id='McAuley-Lab/Amazon-C4',
    filename='sampled_item_metadata_1M.jsonl',
    repo_type='dataset'
)

# Read JSONL file line-by-line
item_pool = []
with open(filepath, 'r') as file:
    for line in file:
        item_pool.append(json.loads(line.strip()))

# Check sample
item_pool[0]


sampled_item_metadata_1M.jsonl:   0%|          | 0.00/643M [00:00<?, ?B/s]

{'item_id': 'B0778XR2QM',
 'category': 'Care',
 'metadata': 'Supergoop! Super Power Sunscreen Mousse SPF 50, 7.1 Fl Oz. Product Description Kids, moms, and savvy sun-seekers will flip for this whip! Formulated with nourishing Shea butter and antioxidant packed Blue Sea Kale, this one-of-a kind mousse formula is making sunscreen super FUN! The refreshing light essence of cucumber and citrus has become an instant hit at Super goop! HQ where we’ve been known to apply gobs of it just for the uplifting scent. Water resistant for up to 80 minutes too! Brand Story Supergoop! is the first and only prestige skincare brand completely dedicated to sun protection. Supergoop! has Super Broad Spectrum protection, which means it protects skin from UVA rays, UVB rays and IRA rays.'}

In [9]:
import pandas as pd

# Convert item metadata list → DataFrame
items_df = pd.DataFrame(item_pool)

print("Items DataFrame shape:", items_df.shape)
items_df.head()


Items DataFrame shape: (1058417, 3)


,item_id,category,metadata
0,B0778XR2QM,Care,Supergoop! Super Power Sunscreen Mousse SPF 50...
1,B07NRD63N7,Clothing,Skirt Sports Women's Kelly Bra. Skirt Sports -...
2,B07Q443QPB,Care,Eyelash Growth Serum By B Radiant - Rapid Lash...
3,B07T589YKW,Clothing,Kamots Beauty High Waist Leggings for Women - ...
4,B09655QKSN,Garden,"Glass Plant Mister Spray Bottle,6.5 inch Vinta..."


In [12]:
import re

def extract_title(text):
    # Title is before first period or first sentence break
    if not isinstance(text, str):
        return None
    return text.split(".")[0].strip()

def extract_description(text):
    if not isinstance(text, str):
        return None
    # Everything after the title
    parts = text.split(".", 1)
    return parts[1].strip() if len(parts) > 1 else None

def extract_brand_from_title(title):
    if not isinstance(title, str):
        return None
    
    # Take first chunk before comma or dash
    brand_candidate = re.split(r'[,\-]| \d', title)[0].strip()
    
    # Clean overly short values
    if len(brand_candidate) < 3:
        return None
    
    return brand_candidate

# Apply corrected brand extraction
items_df["brand"] = items_df["title"].apply(extract_brand_from_title)

# Check again
items_df[["item_id", "category", "title", "brand"]].head()


# Apply extraction
items_df["title"] = items_df["metadata"].apply(extract_title)
items_df["description"] = items_df["metadata"].apply(extract_description)
items_df["brand"] = items_df["title"].apply(extract_brand_from_title)

# Clean category (title case)
items_df["category"] = items_df["category"].astype(str).str.title()

# Check result
items_df[["item_id", "category", "title", "brand"]].head()


,item_id,category,title,brand
0,B0778XR2QM,Care,"Supergoop! Super Power Sunscreen Mousse SPF 50, 7",Supergoop! Super Power Sunscreen Mousse SPF
1,B07NRD63N7,Clothing,Skirt Sports Women's Kelly Bra,Skirt Sports Women's Kelly Bra
2,B07Q443QPB,Care,Eyelash Growth Serum By B Radiant - Rapid Lash...,Eyelash Growth Serum By B Radiant
3,B07T589YKW,Clothing,Kamots Beauty High Waist Leggings for Women - ...,Kamots Beauty High Waist Leggings for Women
4,B09655QKSN,Garden,"Glass Plant Mister Spray Bottle,6",Glass Plant Mister Spray Bottle


In [13]:
# Ensure output folder exists
import os
os.makedirs("data", exist_ok=True)

# Save to CSV
items_df.to_csv("data/cleaned_items.csv", index=False)

# Save to Parquet (recommended for BigQuery)
items_df.to_parquet("data/cleaned_items.parquet", index=False)

print("Saved: cleaned_items.csv and cleaned_items.parquet")


Saved: cleaned_items.csv and cleaned_items.parquet


In [14]:
from google.cloud import bigquery

project_id = "linear-theater-436300-r9"
dataset_id = "ecommerce_pipeline"
table_id = "cleaned_items"

table_full_id = f"{project_id}.{dataset_id}.{table_id}"

# BigQuery client
client = bigquery.Client()

# Load job configuration
job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.PARQUET,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
)

# Load Parquet file into BigQuery
with open("data/cleaned_items.parquet", "rb") as f:
    load_job = client.load_table_from_file(
        f,
        table_full_id,
        job_config=job_config
    )

load_job.result()  # Waits for the job to finish

# Confirm table created
table = client.get_table(table_full_id)
print("Loaded rows:", table.num_rows)
print("Columns:", [schema.name for schema in table.schema])


Loaded rows: 1058417
Columns: ['item_id', 'category', 'metadata', 'title', 'description', 'brand']


## Review data used for: Ranking Agent Evaluation, Retrieval Agent Testing, Orchestrator Testing, Human-AI Collaboration Examples.

In [1]:
from huggingface_hub import hf_hub_download
import pandas as pd

# Download Amazon-C4 query–item pairs
queries_path = hf_hub_download(
    repo_id='McAuley-Lab/Amazon-C4',
    filename='test.csv',
    repo_type='dataset'
)

# Load CSV into pandas
queries_df = pd.read_csv(queries_path)

print("Queries DataFrame shape:", queries_df.shape)
queries_df.head()


Queries DataFrame shape: (21223, 6)


,qid,query,item_id,user_id,ori_rating,ori_review
0,0,I need filters that effectively trap dust and ...,B0C5QYYHTJ,AGREO2G3GTRNYOJK4CIQV2DTZLSQ,5,These filters work I could not believe the amo...
1,1,I need to find a protein that is super healthy...,B0C7D3VLXW,AFLIZT24MDW4XG4HBYKOI3BZGDHQ,5,Love We love this protein we’ve been using it ...
2,2,I need a pillow that helps keep my nasal pillo...,B0C3QRMPVN,AFCSK3W3GI7PGT4655HHKZ2CFFMA,5,CPAP help I use this pillow nightly with my CP...
3,3,I need a memory stick that is excellent and ex...,B0BC13TQJQ,AGGOTCPWSQFI5YHNDLNZ63ABWZVA,5,Excellent. More than expected Memory stick is ...
4,4,I want to buy something that my children will ...,B07Z86PHP8,AEIIPB3DNXXLZX4VRCSVREXGCXUA,5,My children love these! My son asked me to pur...


In [2]:
import os
os.makedirs("data", exist_ok=True)

# Save CSV
queries_df.to_csv("data/queries_c4.csv", index=False)

# Save Parquet (preferred for BigQuery)
queries_df.to_parquet("data/queries_c4.parquet", index=False)

print("Saved queries_c4.csv and queries_c4.parquet")


Saved queries_c4.csv and queries_c4.parquet


In [3]:
from google.cloud import bigquery

project_id = "linear-theater-436300-r9"
dataset_id = "ecommerce_pipeline"
table_id = "raw_queries_c4"

table_full_id = f"{project_id}.{dataset_id}.{table_id}"

client = bigquery.Client()

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.PARQUET,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
)

# Upload Parquet file to BigQuery
with open("data/queries_c4.parquet", "rb") as f:
    load_job = client.load_table_from_file(f, table_full_id, job_config=job_config)

load_job.result()

# Confirm table created
table = client.get_table(table_full_id)
print("Loaded rows:", table.num_rows)
print("Columns:", [schema.name for schema in table.schema])


Loaded rows: 21223
Columns: ['qid', 'query', 'item_id', 'user_id', 'ori_rating', 'ori_review']


## Review data: used for Sentiment analysis

In [4]:
from datasets import load_dataset
import pandas as pd

# Load the dataset in streaming mode (avoids kernel crashes)
stream_ds = load_dataset("amazon_polarity", split="train", streaming=True)

# Take first 50,000 samples
sample_size = 50000
records = []
for i, sample in enumerate(stream_ds):
    if i >= sample_size:
        break
    records.append(sample)

# Convert list → DataFrame
sentiment_df = pd.DataFrame(records)

print("Sentiment dataset shape:", sentiment_df.shape)
sentiment_df.head()


Sentiment dataset shape: (50000, 3)


,label,title,content
0,1,Stuning even for the non-gamer,This sound track was beautiful! It paints the ...
1,1,The best soundtrack ever to anything.,I'm reading a lot of reviews saying that this ...
2,1,Amazing!,This soundtrack is my favorite music of all ti...
3,1,Excellent Soundtrack,I truly like this soundtrack and I enjoy video...
4,1,"Remember, Pull Your Jaw Off The Floor After He...","If you've played the game, you know how divine..."


In [6]:
import os
os.makedirs("data", exist_ok=True)

# Save CSV
sentiment_df.to_csv("data/sentiment_50k.csv", index=False)

# Save Parquet (recommended for BigQuery)
sentiment_df.to_parquet("data/sentiment_50k.parquet", index=False)

print("Saved: sentiment_50k.csv and sentiment_50k.parquet")


Saved: sentiment_50k.csv and sentiment_50k.parquet


In [7]:
from google.cloud import bigquery

project_id = "linear-theater-436300-r9"
dataset_id = "ecommerce_pipeline"
table_id = "raw_reviews_sentiment"

table_full_id = f"{project_id}.{dataset_id}.{table_id}"

client = bigquery.Client()

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.PARQUET,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
)

# Upload Parquet file to BigQuery
with open("data/sentiment_50k.parquet", "rb") as f:
    load_job = client.load_table_from_file(
        f, table_full_id, job_config=job_config
    )

load_job.result()  # Wait for completion

# Confirm table details
table = client.get_table(table_full_id)
print("Loaded rows:", table.num_rows)
print("Columns:", [schema.name for schema in table.schema])


Loaded rows: 50000
Columns: ['label', 'title', 'content']
